# Лабораторна робота №3
## Автоматизація навчання YOLO-моделі за допомогою Python-скриптів

У цьому ноутбуці інтерактивно відтворено конвеєр автоматизованого навчання:
навчання моделі, аналіз метрик, інференс та підготовка артефактів для DVC.

## 1. Імпорт бібліотек і перевірка середовища

In [ ]:
import sys, torch
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print('Project root:', ROOT)

## 2. Параметри експерименту

Для демонстрації на CPU використовуємо малий датасет `construction-ppe-small`,
зменшений розмір зображення та кілька епох.

In [ ]:
DATA = ROOT / 'datasets/construction-ppe-small/data.yaml'
MODEL = 'yolo11n.pt'
EPOCHS = 10
IMGSZ = 320
BATCH = 4
DEVICE = 'cpu'
RUN_NAME = 'yolo11n_construction_ppe_small_nb'
assert DATA.exists(), DATA
print('data.yaml:', DATA)

## 3. Навчання моделі

Виклик `model.train()` з тими самими аргументами, що й у скрипті `train_yolo.py`.

In [ ]:
model = YOLO(MODEL)
results = model.train(data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                      device=DEVICE, project='runs/lab03_notebook', name=RUN_NAME,
                      patience=10, exist_ok=True)
run_dir = Path(model.trainer.save_dir)
print('Run dir:', run_dir)

## 4. Аналіз метрик навчання

Зчитуємо `results.csv` і знаходимо найкращу епоху за `mAP50-95`.

In [ ]:
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [c.strip() for c in df.columns]
map_col = 'metrics/mAP50-95(B)'
best = df.loc[df[map_col].idxmax()]
print('Best epoch:', int(best['epoch']))
print('mAP50-95:', round(float(best[map_col]), 4))
print('mAP50:', round(float(best['metrics/mAP50(B)']), 4))
print('Precision:', round(float(best['metrics/precision(B)']), 4))
print('Recall:', round(float(best['metrics/recall(B)']), 4))
df[['epoch', map_col]].tail()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df['epoch'], df['metrics/precision(B)'], 'o-', label='Precision')
ax[0].plot(df['epoch'], df['metrics/recall(B)'], 's-', label='Recall')
ax[0].set_title('Precision / Recall'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(df['epoch'], df['metrics/mAP50(B)'], 'o-', label='mAP@50')
ax[1].plot(df['epoch'], df[map_col], 's-', label='mAP@50-95')
ax[1].set_title('mAP'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Інференс навченою моделлю

Перевіряємо модель на валідаційних зображеннях.

In [ ]:
best_model = run_dir / 'weights' / 'best.pt'
val_images = ROOT / 'datasets/construction-ppe-small/images/val'
pred_model = YOLO(str(best_model))
preds = pred_model.predict(source=str(val_images), conf=0.25, save=True,
                           project='runs/lab03_notebook_pred', name='predictions', exist_ok=True)
total = sum(len(p.boxes) for p in preds)
print('Images:', len(preds), '| Detected objects:', total)

## 6. Висновок

Відтворено повний цикл: автоматизоване навчання → аналіз метрик → інференс.
Ті самі кроки винесено в скрипти `train_yolo.py`, `analyze_metrics.py`, `predict_yolo.py`,
`prepare_artifacts.py`, а артефакти версіонуються через DVC.